# Tableau metadata documentation generator — Version 10 (Jupyter-safe)

Resolves standard `[Calculation_ID]` and filter-specific `[none:Calculation_ID:nk]` / `[usr:Calculation_ID:nk]` to extracted logical captions. Original source metadata is unchanged.


Physical BigQuery project and dataset are now resolved from the nested TWB connections in the supplied ZIP. No additional credentials are required.

In [ ]:
%pip install pandas python-docx beautifulsoup4

## Pipeline implementation

In [ ]:
"""Offline Tableau metadata -> Confluence-ready DOCX + HTML + JSON manifest.
Requires: pandas, python-docx, beautifulsoup4, nbformat (only for notebook authoring).
No network requests or authentication. Run: python tableau_documentation_pipeline.py --input HKSE.zip
"""
from __future__ import annotations
import argparse, email, hashlib, html, json, re, zipfile, unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

REDACT_KEYS = ('password','secret','token','credential','auth','username','email','private_key','client_id')
SENSITIVE_PATTERN = re.compile(r'(?i)(?:[\w.+-]+@[\w.-]+\.[a-z]{2,}|(?:pat|token|secret|password)\s*[:=]\s*\S+)')
# Presentation-only datasource ID resolver. The raw metadata stays unchanged.
DISPLAY_DATASOURCE_NAMES = {}
FEDERATED_TOKEN = re.compile(r'(?i)federated\.[a-z0-9]+')

def configure_datasource_labels(workbook_datasources, cloud_datasources):
    global DISPLAY_DATASOURCE_NAMES
    labels = {}
    for rec in records(workbook_datasources):
        internal = str(rec.get('name', '')).strip()
        caption = str(rec.get('caption', '')).strip()
        if internal.lower().startswith('federated.') and caption:
            labels[internal.lower()] = caption
    DISPLAY_DATASOURCE_NAMES = labels
    return labels

def display_value(value):
    """Resolve embedded IDs (including [federated.ID].[field]) for report display."""
    raw = str(value)
    def replace(match):
        token = match.group(0)
        return DISPLAY_DATASOURCE_NAMES.get(token.lower(), '[Unresolved Tableau data source]')
    return display_field_value(FEDERATED_TOKEN.sub(replace, raw))

# Presentation-only logical field resolver. Source CSVs/formulas remain unchanged.
DISPLAY_FIELD_NAMES = {}
AMBIGUOUS_FIELD_NAMES = set()
FIELD_TOKEN = re.compile(r'\[[^\]\r\n]+\]')
# Tableau filter column syntax: [none:Calculation_123:nk] / [usr:Calculation_123:nk].
# The middle token is a workbook field ID, not the display caption.
FILTER_CALC_TOKEN = re.compile(r'\[(?:none|usr|sum|avg|count|attr|min|max|median|stdev|stdevp|var|varp):(?P<id>Calculation_\d+):(?P<suffix>[^\]\r\n]+)\]', re.IGNORECASE)
FILTER_RESOLUTION_STATS = {'matched': 0, 'unresolved': 0}


def configure_field_labels(workbook_fields, workbook_calculations):
    """Only map an internal field ID when its nonempty captions agree globally.

    Calculations lack datasource scope in this export, so conflicting captions
    cannot be resolved safely inside free-text expressions.
    """
    global DISPLAY_FIELD_NAMES, AMBIGUOUS_FIELD_NAMES
    candidates = {}
    for df in (workbook_fields, workbook_calculations):
        for rec in records(df):
            name = str(rec.get('name') or '').strip()
            caption = str(rec.get('caption') or '').strip()
            if not name or not caption or caption.lower() in ('nan', 'none'):
                continue
            if not name.startswith('[') or not name.endswith(']'):
                continue
            if name[1:-1].strip().casefold() == caption.casefold():
                continue
            candidates.setdefault(name.casefold(), set()).add(caption)
    DISPLAY_FIELD_NAMES = {k: next(iter(v)) for k, v in candidates.items() if len(v) == 1}
    AMBIGUOUS_FIELD_NAMES = {k for k, v in candidates.items() if len(v) > 1}
    return DISPLAY_FIELD_NAMES

def display_field_value(value):
    """Replace exact bracketed internal tokens, never substrings of field names."""
    raw = str(value)
    def replace(match):
        token = match.group(0)
        caption = DISPLAY_FIELD_NAMES.get(token.casefold())
        if caption is None:
            return token
        return '[' + caption.replace(']', ']]') + ']'
    def replace_filter(match):
        # Lookup uses the same globally unambiguous caption mapping as normal fields.
        internal = '[' + match.group('id') + ']'
        caption = DISPLAY_FIELD_NAMES.get(internal.casefold())
        if caption is None:
            FILTER_RESOLUTION_STATS['unresolved'] += 1
            return match.group(0)
        FILTER_RESOLUTION_STATS['matched'] += 1
        return '[' + caption.replace(']', ']]') + ']'
    # Resolve special Tableau filter wrappers first, then ordinary [Calculation_ID].
    return FILTER_CALC_TOKEN.sub(replace_filter, FIELD_TOKEN.sub(replace, raw))

SECTION_NAMES = [
 'Business Purpose','Dashboard Overview','KPI Definitions','Dashboard Pages',
 'Data Sources','Tableau Data Model','Data Validation','Security & Access',
 'Assumptions & Business Rules','Known Limitations','Support & Contacts','Change Log'
]

def safe(value, limit=900):
    if value is None: return ''
    try:
        if pd.isna(value): return ''
    except (TypeError, ValueError): pass
    text = re.sub(r'\s+', ' ', display_value(value)).strip()
    return SENSITIVE_PATTERN.sub('[REDACTED]', text)[:limit] if limit is not None else SENSITIVE_PATTERN.sub('[REDACTED]', text)

def records(df):
    return df.astype(object).where(pd.notna(df), None).to_dict('records') if not df.empty else []

def field(record, *keys):
    if not isinstance(record,dict): return ''
    lookup={str(k).lower():k for k in record}
    for k in keys:
        actual=lookup.get(k.lower())
        if actual is not None:
            v=safe(record[actual]);
            if v:return v
    return ''

def unzip_safely(path, target):
    target.mkdir(parents=True,exist_ok=True); root=target.resolve()
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            dest=(target/info.filename).resolve()
            if root != dest and root not in dest.parents: raise ValueError('Unsafe archive path: '+info.filename)
            if info.file_size > 100_000_000: raise ValueError('Oversized ZIP entry: '+info.filename)
        z.extractall(target)

def locate(source, output):
    source=Path(source)
    if not source.exists(): raise FileNotFoundError(source)
    root=source
    if source.is_file():
        if source.suffix.lower()!='.zip': raise ValueError('Input must be a ZIP or directory')
        root=output/'_extracted'; unzip_safely(source,root)
    matches=sorted(root.rglob('tableau_metadata_output')) if root.name!='tableau_metadata_output' else [root]
    if matches: return matches[0],root
    if (root/'csv').exists(): return root,root
    raise FileNotFoundError('tableau_metadata_output/csv was not found')

def read_tables(folder):
    tables={}
    for p in sorted((folder/'csv').glob('*.csv')):
        try: tables[p.stem]=pd.read_csv(p,dtype=str,keep_default_na=False,encoding='utf-8-sig',low_memory=False).fillna('')
        except Exception as exc: print('WARNING: could not read',p.name,exc)
    return tables

def preferred(tables,*names):
    for n in names:
        if n in tables and not tables[n].empty:return tables[n]
    return pd.DataFrame()

def reference_outline(root,optional=None):
    candidates=([Path(optional)] if optional else []) + list(root.rglob('*Documentation.doc'))
    for p in candidates:
        if not p.exists():continue
        try:
            from bs4 import BeautifulSoup
            msg=email.message_from_bytes(p.read_bytes())
            for part in msg.walk():
                if part.get_content_type()=='text/html':
                    soup=BeautifulSoup(part.get_payload(decode=True),'html.parser')
                    return [safe(h.get_text(' ',strip=True),140) for h in soup.select('h1,h2,h3,h4')]
        except Exception: pass
    return []

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def style_document(doc):
    sec=doc.sections[0];sec.top_margin=Inches(.72);sec.bottom_margin=Inches(.65)
    sec.left_margin=Inches(.8);sec.right_margin=Inches(.8)
    doc.styles['Normal'].font.name='Aptos';doc.styles['Normal'].font.size=Pt(9)
    for name,size,color in [('Title',23,'17365D'),('Heading 1',15,'17365D'),('Heading 2',11,'2F5597')]:
        st=doc.styles[name];st.font.name='Aptos';st.font.size=Pt(size);st.font.color.rgb=RGBColor.from_string(color)
    footer=sec.footer.paragraphs[0];footer.alignment=WD_ALIGN_PARAGRAPH.CENTER
    footer.add_run('INTERNAL • DRAFT • Metadata-derived technical documentation')

def add_table(doc,columns,rows,max_rows=None):
    columns=list(columns);rows=list(rows) if max_rows is None else list(rows)[:max_rows]
    if not columns:return
    # Number each displayed metadata/item row, restarting at 1 for every table.
    # This is a presentation-only column: source metadata remains unchanged.
    # All report tables, including document control and change log, use this helper.
    if columns[0].strip().lower() != 'no.':
        columns=['No.']+columns
        rows=[[str(i), *list(row)] for i,row in enumerate(rows, start=1)]
    tbl=doc.add_table(rows=1,cols=len(columns));tbl.style='Table Grid'
    for c,name in zip(tbl.rows[0].cells,columns):
        c.text=safe(name,90)
        for r in c.paragraphs[0].runs:r.bold=True;r.font.size=Pt(8)
        shd=OxmlElement('w:shd');shd.set(qn('w:fill'),'DCE8F4');c._tc.get_or_add_tcPr().append(shd)
    for row in rows:
        cells=tbl.add_row().cells
        for c,value in zip(cells,row):
            c.text=safe(value,None)
            for p in c.paragraphs:
                for r in p.runs:r.font.size=Pt(7.5)
    doc.add_paragraph('')

def selected_rows(df,columns,limit=None):
    return [[field(rec,c) if not any(x in c.lower() for x in REDACT_KEYS) else '[REDACTED]' for c in columns]
            for rec in (records(df) if limit is None else records(df)[:limit]) ]

def add_inventory(doc,title,df,columns,limit=None):
    doc.add_heading(title,level=2)
    if df.empty:doc.add_paragraph('Not available in the supplied metadata export.');return
    present=[c for c in columns if c in df.columns]
    if not present:present=[c for c in df.columns if not any(x in c.lower() for x in REDACT_KEYS)][:5]
    add_table(doc,present,selected_rows(df,present,limit),limit)
    # All selected inventory records are included; the exhaustive appendix lists every source table and column.



def normalized_title(value):
    value=unicodedata.normalize('NFKC',str(value)).lower()
    value=re.sub(r'_[0-9a-f]{8}-[0-9a-f]{3,}.*$','',value)
    return re.sub(r'[^a-z0-9]+','',value)

def is_sensitive_col(name):
    name=str(name).lower()
    return any(key in name for key in REDACT_KEYS)

def insert_matching_screenshots(doc, page_title, screenshots, used):
    target=normalized_title(page_title)
    matches=[]
    for shot in screenshots:
        stem=re.sub(r'_[0-9a-f]{8}-[0-9a-f]{3,}.*$','',shot.stem,flags=re.I)
        if normalized_title(stem)==target and str(shot) not in used:
            matches.append(shot)
    for shot in matches:
        doc.add_paragraph('Screenshot evidence: '+shot.stem)
        doc.add_picture(str(shot),width=Inches(6.7))
        used.add(str(shot))

def append_full_catalogue(doc, tables, metadata_dir):
    doc.add_page_break()
    doc.add_heading('Appendix B. Complete Metadata Record Catalogue',level=1)
    doc.add_paragraph('Every row from every CSV in the supplied metadata export is represented below, including the change log. The original source row order is retained. All non-sensitive column values are included without character truncation; credential/authentication-related columns are redacted. Raw XML is included unless it contains detected sensitive content. The catalogue may be long and slow to edit in Microsoft Word.')
    catalogue=dict(tables)
    change_log=metadata_dir/'history'/'tableau_metadata_change_log.csv'
    if change_log.exists():
        catalogue['tableau_metadata_change_log']=pd.read_csv(change_log,dtype=str,keep_default_na=False,encoding='utf-8-sig',low_memory=False)
    for name,df in sorted(catalogue.items()):
        doc.add_heading(f'B. {name} — {len(df):,} records',level=2)
        if df.empty:
            doc.add_paragraph('0 records in source CSV.')
            continue
        cols=list(df.columns)
        doc.add_paragraph('Source columns: '+', '.join(cols))
        for index,rec in enumerate(records(df),1):
            pieces=[]
            for col in cols:
                val='[REDACTED: sensitive column]' if is_sensitive_col(col) else safe(rec.get(col),None)
                pieces.append(f'{col}={val}')
            para=doc.add_paragraph(style='Normal')
            para.paragraph_format.space_after=Pt(2)
            para.paragraph_format.keep_together=False
            run=para.add_run(f'{index:05d} | ')
            run.bold=True
            run.font.size=Pt(7)
            run=para.add_run(' | '.join(pieces))
            run.font.size=Pt(7)

def extract_physical_connections(root):
    """Read nested physical connections from a TWBX/TWB in the input archive.

    Prefer the Google BigQuery connection over Tableau's federated logical
    wrapper and Hyper extract. Never return authentication or credential data.
    """
    candidates = sorted(root.rglob('*.twbx')) + sorted(root.rglob('*.twb'))
    for workbook in candidates:
        try:
            if workbook.suffix.lower() == '.twbx':
                with zipfile.ZipFile(workbook) as archive:
                    members = [n for n in archive.namelist() if n.lower().endswith('.twb')]
                    if not members: continue
                    xml_bytes = archive.read(members[0])
            else:
                xml_bytes = workbook.read_bytes()
            xml_root = ET.fromstring(xml_bytes)
            found = {}
            for ds in xml_root.findall('./datasources/datasource'):
                internal = ds.get('name', '')
                if not internal: continue
                nested = ds.findall('.//connection')
                physical = [c for c in nested if c.get('class', '').lower() not in ('federated', 'hyper')]
                if not physical: continue
                # A data source can contain multiple physical connections.
                # Use the first BigQuery connection, otherwise the first physical one.
                chosen = next((c for c in physical if c.get('class','').lower() == 'googlebigquery'), physical[0])
                kind = chosen.get('class','').lower()
                db = chosen.get('dbname','').strip()
                schema = chosen.get('schema','').strip()
                server = chosen.get('server','').strip()
                # BigQuery has no ordinary customer-managed SQL server hostname.
                server_display = 'Not applicable — Google BigQuery cloud service' if kind == 'googlebigquery' else (server or '[NOT AVAILABLE]')
                found[internal] = {
                    'Server': server_display,
                    'Database': db or '[NOT AVAILABLE]',
                    'Schema': schema or '[NOT AVAILABLE]',
                    'Physical connector': kind or '[NOT AVAILABLE]',
                    'Metadata source': 'Nested physical connection in packaged TWB',
                }
            return found
        except (OSError, zipfile.BadZipFile, ET.ParseError, KeyError) as exc:
            print('WARNING: could not inspect workbook connections:', workbook.name, exc)
    return {}


def resolve_connection_names(connections, workbook_datasources, cloud_datasources, physical_connections=None):
    """Resolve readable Tableau captions and safe nested physical connection metadata."""
    physical_connections = physical_connections or {}
    columns = ['Data source name','Tableau internal ID','Connection layer','Cloud data source',
               'Server','Database','Schema','Metadata status']
    if connections.empty:
        return pd.DataFrame(columns=columns)
    wb_lookup = {str(r.get('name','')).strip():field(r,'caption') for r in records(workbook_datasources) if str(r.get('name','')).strip()}
    cloud_names = [field(r,'name') for r in records(cloud_datasources) if field(r,'name')]
    norm = lambda v: re.sub(r'[^a-z0-9]+','',str(v).lower())
    cloud_index = {norm(name):name for name in cloud_names}
    rows = []
    for r in records(connections):
        internal = str(r.get('datasource','')).strip()
        caption = wb_lookup.get(internal,'')
        cloud_match = cloud_index.get(norm(caption),'') if caption else ''
        physical = physical_connections.get(internal,{})
        has_physical = bool(physical)
        rows.append({
            'Data source name': caption or cloud_match or '[UNRESOLVED — review workbook metadata]',
            'Tableau internal ID': '[Technical ID retained in source CSV]',
            'Connection layer': (field(r,'connection_type') or '[NOT EXPORTED]') +
                (' → '+physical['Physical connector'] if has_physical else ''),
            'Cloud data source': cloud_match or '[NOT MATCHED]',
            'Server': physical.get('Server') or field(r,'server') or '[NOT EXPORTED]',
            'Database': physical.get('Database') or field(r,'database') or '[NOT EXPORTED]',
            'Schema': physical.get('Schema') or field(r,'schema') or '[NOT EXPORTED]',
            'Metadata status': 'Physical connection resolved from TWB' if has_physical else
                               ('Logical caption resolved; physical connection not available' if caption else 'Unresolved internal ID'),
        })
    return pd.DataFrame(rows)

def run(input_path,output_dir='tableau_documentation_output',template_path=None,report_title=None):
    out=Path(output_dir).resolve();out.mkdir(parents=True,exist_ok=True)
    md,root=locate(input_path,out);t=read_tables(md)
    configure_datasource_labels(preferred(t,'twb_datasources'), preferred(t,'cloud_datasources'))
    configure_field_labels(preferred(t,'twb_fields'), preferred(t,'twb_calculations'))
    FILTER_RESOLUTION_STATS.update(matched=0, unresolved=0)
    wb=preferred(t,'cloud_workbook');wbr=records(wb)[0] if not wb.empty else {}
    title=report_title or field(wbr,'name') or 'Tableau Workbook'
    slug=re.sub(r'[^a-zA-Z0-9_-]+','_',title).strip('_')[:75]
    project=field(wbr,'projectName','containerName') or 'To be confirmed'
    dashboards=preferred(t,'cloud_dashboards','twb_dashboards')
    sheets=preferred(t,'cloud_sheets','twb_worksheets')
    ds=preferred(t,'cloud_datasources','twb_datasources')
    fields=preferred(t,'cloud_datasource_fields','twb_fields')
    calculations=preferred(t,'twb_calculations','cloud_calculated_fields')
    parameters=preferred(t,'twb_parameters','cloud_parameters')
    filters=preferred(t,'twb_filters')
    zones=preferred(t,'twb_dashboard_zones')
    connections=preferred(t,'twb_connections')
    physical_connections=extract_physical_connections(root)
    resolved_connections=resolve_connection_names(connections,preferred(t,'twb_datasources'),preferred(t,'cloud_datasources'),physical_connections)
    qa=preferred(t,'qa')
    outline=reference_outline(root,template_path)
    screenshots=sorted((md/'view_screenshots').rglob('*.png')) if (md/'view_screenshots').exists() else []
    used_screenshots=set()
    generated=datetime.now(timezone.utc).isoformat()
    doc=Document();style_document(doc)
    p=doc.add_paragraph(style='Title');p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.add_run(title)
    p=doc.add_paragraph('TABLEAU DASHBOARD DOCUMENTATION');p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph('Document status: DRAFT — technical inventory generated automatically; business validation required.')
    doc.add_paragraph('Generated (UTC): '+generated)
    doc.add_page_break()
    doc.add_heading('Document Control',level=1)
    add_table(doc,['Property','Value'],[
        ('Report',title),('Tableau project',project),('Workbook LUID',field(wbr,'luid')),
        ('Extraction run ID',field(wbr,'run_id')),('Generation time (UTC)',generated),
        ('Report owner','[OWNER TO CONFIRM]'),('Business approver','[TO CONFIRM]'),
        ('Document status','Draft for review'),('Source','Supplied Tableau metadata ZIP; no live Tableau query')])
    doc.add_heading('Executive Technical Summary',level=1)
    add_table(doc,['Metadata artifact','Records'],[(name,str(len(df))) for name,df in [
        ('Dashboards',dashboards),('Worksheets',sheets),('Data sources',ds),('Fields',fields),
        ('Calculated fields',calculations),('Parameters',parameters),('Filters',filters),
        ('Dashboard layout zones',zones),('Connections',connections)]])
    doc.add_heading('1. Business Purpose',level=1)
    for k in ['Objective','Business Value','Target Audience']:
        doc.add_heading(k,level=2);doc.add_paragraph('[BUSINESS OWNER TO CONFIRM] The metadata alone does not establish this information.')
    doc.add_heading('2. Dashboard Overview',level=1)
    doc.add_paragraph(f'Tableau workbook: {title}. Project: {project}.')
    doc.add_paragraph('The dashboard and worksheet inventories below were extracted from Tableau metadata. Confirm the report narrative, refresh SLA, and intended audience.')
    add_inventory(doc,'Dashboards',dashboards,['name','dashboard_luid','path','createdAt','updatedAt'],60)
    add_inventory(doc,'Worksheets',sheets,['name','path','sheet_luid'],75)
    doc.add_heading('3. KPI Definitions',level=1)
    doc.add_paragraph('Candidate KPIs and technical formulas are listed below. These are NOT approved business definitions. KPI purpose, aggregation grain, currency, and filter behavior require owner validation.')
    add_inventory(doc,'Calculated-field formula inventory',calculations,['caption','name','datatype','role','formula'],65)
    doc.add_heading('4. Dashboard Pages',level=1)
    doc.add_paragraph('Each page corresponds to a dashboard found in the export. Worksheet-to-dashboard associations are inferred from dashboard layout zones where available; verify against Tableau UI.')
    dash_names=list(dict.fromkeys([field(r,'name') for r in records(dashboards) if field(r,'name')]))
    if not dash_names:doc.add_paragraph('No dashboard names were present in the export.')
    for idx,dash in enumerate(dash_names,1):
        doc.add_heading(f'Page {idx}: {dash}',level=2)
        doc.add_paragraph('Purpose: [BUSINESS OWNER TO CONFIRM]')
        zone_rows=[r for r in records(zones) if field(r,'dashboard')==dash]
        views=list(dict.fromkeys([field(r,'name') for r in zone_rows if field(r,'name')]))
        doc.add_paragraph('Layout components / view references: '+(', '.join(views[:25]) if views else 'Not resolved from metadata.'))
        doc.add_paragraph('Visualization interpretation and KPI mapping: [REPORT OWNER TO CONFIRM]')
        insert_matching_screenshots(doc, dash, screenshots, used_screenshots)
    if screenshots and len(used_screenshots)<len(screenshots):
        doc.add_heading('Unmatched Screenshot Evidence',level=2)
        for shot in screenshots:
            if str(shot) not in used_screenshots:
                doc.add_paragraph('No dashboard-title match: '+shot.stem)
                doc.add_picture(str(shot),width=Inches(6.7))
                used_screenshots.add(str(shot))
    doc.add_heading('5. Data Sources',level=1)
    add_inventory(doc,'Published / logical data sources',ds,['name','hasExtracts','hasUserReference','containsUnsupportedCustomSql'],60)
    add_inventory(doc,'Connection metadata (secrets excluded)',resolved_connections,['Data source name','Tableau internal ID','Connection layer','Cloud data source','Server','Database','Schema','Metadata status'],50)
    doc.add_paragraph('Note: federated.* is the Tableau logical connection layer. Physical connection details are resolved from the nested Google BigQuery connection in the packaged TWB when available. For Google BigQuery, Database denotes the GCP project and Schema denotes the BigQuery dataset; a conventional database server hostname is not applicable. The Hyper connection represents the Tableau extract, not the upstream BigQuery database. Credentials are never included. If no packaged workbook is present, unavailable values remain marked NOT EXPORTED.')
    doc.add_paragraph('Refresh frequency and data freshness SLA: [PLATFORM OWNER TO CONFIRM].')
    doc.add_heading('6. Tableau Data Model',level=1)
    doc.add_paragraph('Field dictionary and connection metadata are evidence of available objects, but do not independently establish physical joins or relationship cardinalities.')
    add_inventory(doc,'Field dictionary',fields,['datasource_name','field_name','description','role','dataType'],90)
    add_inventory(doc,'Workbook-defined fields',preferred(t,'twb_fields'),['datasource','caption','name','datatype','role','aggregation','hidden'],55)
    doc.add_paragraph('Join logic, table grain, and row-level uniqueness: [DATA ENGINEER TO CONFIRM].')
    doc.add_heading('7. Data Validation',level=1)
    add_inventory(doc,'Extractor QA checks',qa,['check','status','detail'],40)
    doc.add_paragraph('Recommended acceptance tests: reconcile source-to-Tableau totals; test filter selections; check nulls and duplication; validate period comparisons and currencies; verify refresh completeness. These tests have not been executed by this generator.')
    doc.add_heading('8. Security & Access',level=1)
    doc.add_paragraph('Confirm workbook/project permissions, row-level security, data classifications, published connection permissions, and credential ownership directly in Tableau Cloud. No credentials are exported into this document.')
    doc.add_heading('9. Assumptions & Business Rules',level=1)
    doc.add_paragraph('Included: technical field names, formulas, workbook inventory and exported metadata as observed.')
    doc.add_paragraph('Excluded: undocumented KPI semantics, unsupported claims about data freshness, inferred SQL joins, or claims about access rights.')
    add_inventory(doc,'Parameters',parameters,['caption','name','datatype','param_domain_type','value'],65)
    add_inventory(doc,'Filter definitions',filters,['filter_type','column','member','formula','parent_tag','parent_name'],65)
    doc.add_heading('10. Known Limitations',level=1)
    for issue in [
        'This is a metadata-derived draft, not a runtime validation or certified business specification.',
        'Some metadata exports may contain incomplete dashboard relationships, lineage, formulas or SQL.',
        'Calculated-field definitions may be duplicated across embedded and published data sources.',
        'Sensitive attributes and authentication-related fields are omitted or redacted.',
        'The Word report contains the existing curated technical sections only; full source metadata remains available in the original CSV and JSON exports.',
        'A business owner must approve business purpose, KPI definitions, refresh frequency and audience.']:
        doc.add_paragraph(issue,style='List Bullet')
    doc.add_heading('11. Support & Contacts',level=1)
    add_table(doc,['Role','Contact'],[('Business owner','[TO CONFIRM]'),('Tableau platform owner','[TO CONFIRM]'),('Data engineering support','[TO CONFIRM]'),('Report maintainer','[TO CONFIRM]')])
    doc.add_heading('12. Change Log',level=1)
    add_table(doc,['Date (UTC)','Version','Change','Status'],[(generated[:10],'1.0','Initial documentation generated from Tableau metadata','Draft')])
    docx=out/(slug+'_Technical_Documentation.docx');doc.save(docx)
    # Companion HTML: same major sections, plus fuller searchable inventories.
    parts=[f'<h1>{html.escape(title)}</h1><p>Tableau Dashboard Documentation | Draft | {html.escape(generated)}</p>',
           '<p>Business purpose, KPI definitions and access details require owner validation.</p>']
    for idx,section in enumerate(SECTION_NAMES,1):
        parts.append(f'<h2>{idx}. {html.escape(section)}</h2>')
        if section=='Business Purpose':parts.append('<p>Objective, Business Value, Target Audience: [BUSINESS OWNER TO CONFIRM]</p>')
        elif section=='Dashboard Overview':parts.append(dashboards.head(100).to_html(index=False,escape=True))
        elif section=='KPI Definitions':parts.append(calculations[[c for c in ['caption','name','datatype','role','formula'] if c in calculations]].head(250).to_html(index=False,escape=True))
        elif section=='Dashboard Pages':parts.append(zones[[c for c in ['dashboard','name','type','x','y','w','h'] if c in zones]].head(350).to_html(index=False,escape=True))
        elif section=='Data Sources':parts.append(ds.head(100).to_html(index=False,escape=True))
        elif section=='Tableau Data Model':parts.append(fields[[c for c in ['datasource_name','field_name','description','role','dataType'] if c in fields]].head(350).to_html(index=False,escape=True))
        elif section=='Data Validation':parts.append(qa.to_html(index=False,escape=True))
        elif section=='Assumptions & Business Rules':parts.append(parameters[[c for c in ['caption','name','datatype','value'] if c in parameters]].head(120).to_html(index=False,escape=True))
        else:parts.append('<p>[REVIEW AND CONFIRM WITH REPORT OWNER]</p>')
    parts.append('<h2>Appendix: Workbook Filters</h2>'+filters[[c for c in ['filter_type','column','member','formula','parent_tag','parent_name'] if c in filters]].to_html(index=False,escape=True))
    for table_name, table_df in sorted(t.items()):
        parts.append('<h2>Complete source table: '+html.escape(table_name)+'</h2>')
        parts.append(table_df[[c for c in table_df.columns if not is_sensitive_col(c)]].to_html(index=False,escape=True) if not table_df.empty else '<p>0 records</p>')
    css='body{font:14px/1.5 Arial,sans-serif;margin:3em auto;max-width:1150px;padding:0 20px;color:#243041}h1,h2{color:#17365d}table{border-collapse:collapse;font-size:11px;width:100%;margin:1em 0 2em}th,td{border:1px solid #ccd5e0;padding:6px;overflow-wrap:anywhere}th{background:#e4edf7}tr:nth-child(even){background:#f7f9fc}'
    html_file=out/(slug+'_Technical_Documentation.html')
    for shot in screenshots:
        import base64
        encoded=base64.b64encode(shot.read_bytes()).decode('ascii')
        parts.append('<h2>Screenshot: '+html.escape(shot.stem)+'</h2><img style="max-width:100%" src="data:image/png;base64,'+encoded+'"/>')
    html_file.write_text('<!doctype html><html><head><meta charset="utf-8"><style>'+css+'</style></head><body>'+'\n'.join(parts)+'</body></html>',encoding='utf-8')
    manifest={'generator':'offline_tableau_documentation_pipeline__physical_connections','generated_utc':generated,'input':str(input_path),
        'workbook':title,'project':project,'source_metadata_dir':str(md),
        'counts':{name:len(df) for name,df in t.items()},'reference_headings':outline,
        'numbering':'No. column, restarting at 1 in each Word table','full_record_appendix':False,'word_appendices_included':False,'connection_name_resolution':'twb_connections.datasource -> twb_datasources.name -> twb_datasources.caption','federated_ids_in_word':'resolved to friendly captions for display; raw CSVs unchanged','screenshot_files':[str(x.name) for x in screenshots],
        'screenshots_embedded':len(used_screenshots),
        'physical_connection_resolution': {'datasources_resolved':len(physical_connections),'source':'packaged TWBX/TWB nested physical connections','credentials_excluded':True},
        'logical_field_name_resolution': {'unique_identifiers_resolved':len(DISPLAY_FIELD_NAMES),'ambiguous_identifiers_preserved':len(AMBIGUOUS_FIELD_NAMES),'source_metadata_unchanged':True, 'filter_reference_resolution':dict(FILTER_RESOLUTION_STATS)},
        'outputs':{p.name:{'sha256':sha256(p),'bytes':p.stat().st_size} for p in [docx,html_file]},
        'publication':'Manual only; no Atlassian/Confluence integration','approval':'Owner validation required'}
    manifest_path=out/'report_generation_manifest.json';manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    return {'word':str(docx),'html':str(html_file),'manifest':str(manifest_path),'counts':manifest['counts']}

if False and __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--input',required=True,help='Metadata ZIP or extracted folder')
    parser.add_argument('--output',default='tableau_documentation_output')
    parser.add_argument('--template',default=None,help='Optional Confluence-exported .doc sample')
    parser.add_argument('--title',default=None)
    args=parser.parse_args()
    print(json.dumps(run(args.input,args.output,args.template,args.title),indent=2))

if False and __name__ == '__main__' and not ('ipykernel' in __import__('sys').modules):
    from pathlib import Path
    import argparse
    parser=argparse.ArgumentParser()
    parser.add_argument('input',type=Path,help='HKSE.zip or extracted directory')
    parser.add_argument('--output',type=Path,default=Path('tableau_documentation_output'))
    args=parser.parse_args()
    print(run(args.input,args.output))


"""Offline Tableau metadata explorer generator. No network calls or external JS dependencies."""
from pathlib import Path
import zipfile, tempfile, json, csv, io, html, re, hashlib

TABLES=['cloud_workbook','cloud_dashboards','cloud_sheets','cloud_datasources','cloud_datasource_fields','cloud_worksheet_datasource_fields','twb_datasources','twb_fields','twb_calculations','twb_worksheets','twb_dashboards','twb_connections','twb_custom_sql','twb_filters','twb_dashboard_zones','twb_parameters']
SENSITIVE=re.compile(r'password|secret|token|credential|private.key|authorization|username|email|raw_xml|authentication',re.I)
def build_explorer(input_path,output_dir,report_title=None):
    input_path=Path(input_path);out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    def read_zip(z):
        result={}
        for name in TABLES:
            f='csv/'+name+'.csv'
            if f in z.namelist():
                content=z.read(f).decode('utf-8-sig',errors='replace')
                result[name]=[{k:v for k,v in row.items() if not SENSITIVE.search(k)} for row in csv.DictReader(io.StringIO(content))]
            else:result[name]=[]
        manifest=json.loads(z.read('tableau_metadata_run_manifest.json')) if 'tableau_metadata_run_manifest.json' in z.namelist() else {}
        return result,manifest
    if input_path.is_file() and input_path.suffix.lower()=='.zip':
        with zipfile.ZipFile(input_path) as z:tables,manifest=read_zip(z)
    elif input_path.is_dir():
        tables={}
        for name in TABLES:
            f=input_path/'csv'/(name+'.csv')
            tables[name]=[{k:v for k,v in row.items() if not SENSITIVE.search(k)} for row in csv.DictReader(f.open(encoding='utf-8-sig',newline=''))] if f.exists() else []
        m=input_path/'tableau_metadata_run_manifest.json';manifest=json.loads(m.read_text(encoding='utf-8')) if m.exists() else {}
    else:raise ValueError('Input must be a metadata ZIP or extracted directory')
    workbook=manifest.get('selected_workbook') or {}
    title=report_title or (workbook.get('name') if isinstance(workbook,dict) else None) or manifest.get('workbook_name_requested') or input_path.stem
    nodes=[];edges=[];node_ids={};seen_edges=set()
    def add(kind,key,label,detail=None):
        key=str(key or label);ident=kind+':'+key
        if ident not in node_ids:
            node_ids[ident]=len(nodes);nodes.append({'id':ident,'kind':kind,'label':str(label or key),'detail':detail or {}})
        return ident
    def link(a,b,kind='metadata',confidence='verified',reason=''):
        if not a or not b:return
        k=(a,b,kind)
        if k not in seen_edges:seen_edges.add(k);edges.append({'from':a,'to':b,'kind':kind,'confidence':confidence,'reason':reason})
    wb=add('workbook',title,title)
    for row in tables['twb_dashboards']:
        d=add('dashboard',row.get('name'),row.get('caption') or row.get('name'),row);link(d,wb,'contained-in')
    for row in tables['twb_worksheets']:
        s=add('worksheet',row.get('name'),row.get('caption') or row.get('name'),row);link(s,wb,'contained-in')
    for row in tables['twb_datasources']:
        ds=add('datasource',row.get('name'),row.get('caption') or row.get('name'),row);link(ds,wb,'contained-in')
    for row in tables['twb_fields']:
        dsid='datasource:'+str(row.get('datasource'))
        if dsid not in node_ids:continue
        f=add('field',str(row.get('datasource'))+'::'+str(row.get('name')),row.get('caption') or row.get('name'),row)
        link(f,dsid,'belongs-to')
    for row in tables['cloud_worksheet_datasource_fields']:
        name=row.get('sheet_name');fname=row.get('field_name')
        if not name or not fname:continue
        s='worksheet:'+str(name)
        if s not in node_ids:continue
        # Source does not guarantee a unique datasource for field name; link to a reference node instead.
        f=add('field-reference',str(name)+'::'+str(row.get('field_id') or fname),fname,row)
        link(f,s,'used-by','verified','Tableau Metadata API worksheet field association')
    for row in tables['twb_dashboard_zones']:
        d='dashboard:'+str(row.get('dashboard') or row.get('dashboard_name') or row.get('parent_name'))
        s='worksheet:'+str(row.get('name') or row.get('sheet') or row.get('worksheet'))
        if d in node_ids and s in node_ids:link(s,d,'displayed-in')
    for row in tables['twb_connections']:
        ds='datasource:'+str(row.get('datasource'))
        if ds not in node_ids:continue
        parts=[row.get(k,'') for k in ('server','database','schema')]
        if any(parts):
            label=' / '.join(v for v in parts if v)
            conn=add('connection',label,label,row);link(conn,ds,'connects-to','verified')
    formulas=[]
    for row in tables['twb_calculations']:
        if row.get('formula'):
            formulas.append({'name':row.get('caption') or row.get('name'),'formula':row['formula'],'kind':'Tableau Calculation','source':'twb_calculations'})
    for row in tables['twb_custom_sql']:
        sql=row.get('sql') or row.get('query') or row.get('text') or row.get('custom_sql')
        if sql:formulas.append({'name':row.get('name') or 'Custom SQL','formula':sql,'kind':'Custom SQL','source':'twb_custom_sql'})
    model={'title':title,'nodes':nodes,'edges':edges,'tables':tables,'transformations':formulas,'counts':{k:len(v) for k,v in tables.items()},'manifest':{'workbook':title,'extracted_at':manifest.get('extracted_at_utc') or manifest.get('run_id'),'source':'Workflow 3 metadata ZIP'},'limitations':['BigQuery physical schema and SQL repository lineage not connected','ERD reflects extracted Tableau logical associations, not verified physical primary/foreign keys','Missing connection server/database/schema values are shown as unavailable','No custom SQL found' if not tables['twb_custom_sql'] else 'Custom SQL availability depends on extraction']}
    payload=json.dumps(model,ensure_ascii=False).replace('</','<\\/')
    html_template=r'''<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Tableau Metadata Explorer</title><style>
:root{font-family:Segoe UI,Arial,sans-serif;color:#dce7f5;background:#0b1728}*{box-sizing:border-box}body{margin:0}header{padding:19px 28px;background:#11263d;border-bottom:2px solid #18a4a7}header h1{font-size:21px;margin:0 0 5px}header p{margin:0;color:#9db3c7;font-size:12px}.shell{display:grid;grid-template-columns:245px 1fr;min-height:calc(100vh - 84px)}aside{background:#102238;padding:18px;border-right:1px solid #2c4259}aside button{width:100%;background:transparent;color:#cbd9e9;border:0;padding:12px;text-align:left;border-radius:8px;cursor:pointer;margin:3px 0}aside button.active,aside button:hover{background:#147d83;color:white}main{padding:24px;min-width:0}h2{margin:0 0 15px;font-size:20px}h3{font-size:15px;color:#aee9e9}.card{background:#132a41;border:1px solid #28435d;border-radius:10px;padding:17px;margin-bottom:16px}.metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px}.metric b{display:block;font-size:27px;color:#60d2d0}input,select{background:#0a1b2c;color:white;border:1px solid #3b5971;border-radius:7px;padding:10px;width:100%}table{width:100%;border-collapse:collapse;font-size:12px}th,td{border-bottom:1px solid #30485f;padding:9px;text-align:left;vertical-align:top;word-break:break-word}th{color:#8bdcda;position:sticky;top:0;background:#132a41}section{display:none}section.active{display:block}.scroll{max-height:490px;overflow:auto}.graph{width:100%;height:470px;background:#091a2b;border:1px solid #35546d;border-radius:9px;overflow:auto}svg{width:100%;height:100%;min-width:850px}svg text{fill:white;font-size:11px;pointer-events:none}svg .node{cursor:pointer}svg .node rect{stroke:#6e9aaf;stroke-width:1}svg .node:hover rect{stroke:#fff;stroke-width:2}.toolbar{display:flex;gap:10px;align-items:center;margin-bottom:12px}.toolbar input{max-width:430px}.toolbar select{max-width:190px}pre{white-space:pre-wrap;word-break:break-word;background:#091a2b;border:1px solid #2d4b62;padding:16px;border-radius:8px;color:#aee9e9;max-height:440px;overflow:auto}.muted{color:#a9bed1;font-size:12px}.chip{display:inline-block;border:1px solid #397c85;border-radius:20px;padding:4px 10px;margin:4px;color:#8ce2de;font-size:11px}.notice{border-left:3px solid #e5aa54;padding:9px 13px;background:#302d2d;margin-bottom:12px;font-size:12px}@media(max-width:850px){.shell{display:block}aside{display:flex;flex-wrap:wrap}aside button{width:auto}.metrics{grid-template-columns:repeat(2,1fr)}main{padding:14px}}</style></head><body>
<header><h1>Tableau Metadata Intelligence Explorer</h1><p id="subtitle"></p></header><div class="shell"><aside><button class="active" data-tab="overview">◈ Overview & Discovery</button><button data-tab="lineage">◇ Metadata Lineage</button><button data-tab="mapping">▤ Data Mapping / ERD</button><button data-tab="logic">⌘ Transformation Logic</button><button data-tab="inventory">▦ Source Metadata Inventory</button><button data-tab="quality">⚑ Evidence & Gaps</button></aside><main>
<section id="overview" class="active"><h2>Metadata Discovery</h2><div class="metrics" id="metrics"></div><div class="card"><h3>Search extracted objects</h3><input id="find" placeholder="Search workbook, dashboards, worksheets, fields, data sources..."/><div class="scroll"><table><thead><tr><th>Type</th><th>Object</th><th>Source identifier</th></tr></thead><tbody id="searchResults"></tbody></table></div></div></section>
<section id="lineage"><h2>Metadata Discovery & Lineage</h2><p class="muted">Arrows point from upstream/contained objects toward downstream consumers. Select nodes to inspect provenance. The graph is a documented subset, not a complete BigQuery lineage.</p><div class="toolbar"><select id="graphKind"><option value="all">All object types</option><option value="worksheet">Worksheets</option><option value="datasource">Data sources</option><option value="dashboard">Dashboards</option></select><input id="graphSearch" placeholder="Find node by name"/></div><div class="graph"><svg id="lineageSvg" viewBox="0 0 1000 500"></svg></div><div class="card" id="nodeDetails">Select a node to view metadata.</div></section>
<section id="mapping"><h2>Visual Data Mapping & ERD-style View</h2><div class="notice">Relationships here are metadata associations. Physical database PK/FK constraints and cardinality are not asserted unless independently verified.</div><div class="toolbar"><select id="dsSelect"></select></div><div class="card" id="dsDetails"></div><div class="card"><h3>Fields in selected data source</h3><div class="scroll" id="fieldTable"></div></div></section>
<section id="logic"><h2>Transformation Logic</h2><p class="muted">Original Tableau calculation formulas and available custom SQL. BigQuery SQL, dbt and Dataform source code require future connectors.</p><div class="toolbar"><input id="logicSearch" placeholder="Search calculation names or formulas"/></div><div class="card" id="logicList"></div></section>
<section id="inventory"><h2>Source Metadata Inventory</h2><div class="toolbar"><select id="tableSelect"></select><input id="tableSearch" placeholder="Search records"/></div><div class="card scroll" id="tableResults"></div></section>
<section id="quality"><h2>Provenance, Completeness & Limitations</h2><div id="limitations"></div><div class="card"><h3>Extracted table counts</h3><div id="tableCounts"></div></div></section>
</main></div><script id="data" type="application/json">__DATA__</script><script>
const D=JSON.parse(document.getElementById('data').textContent);document.getElementById('subtitle').textContent=D.title+'  |  Offline snapshot from Workflow 3  |  No live BigQuery connection';
const esc=s=>String(s??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const cap=s=>s.replace(/-/g,' ').replace(/\b\w/g,c=>c.toUpperCase());
function show(tab){document.querySelectorAll('section').forEach(x=>x.classList.toggle('active',x.id===tab));document.querySelectorAll('aside button').forEach(x=>x.classList.toggle('active',x.dataset.tab===tab));if(tab==='lineage')drawGraph();}
document.querySelectorAll('aside button').forEach(b=>b.onclick=()=>show(b.dataset.tab));
const counts=[['Dashboards',D.tables.twb_dashboards.length],['Worksheets',D.tables.twb_worksheets.length],['Data Sources',D.tables.twb_datasources.length],['Calculated Fields',D.transformations.filter(x=>x.kind==='Tableau Calculation').length]];
document.getElementById('metrics').innerHTML=counts.map(([k,v])=>`<div class="card metric"><b>${v}</b>${k}</div>`).join('');
function searchNodes(){let q=document.getElementById('find').value.toLowerCase();document.getElementById('searchResults').innerHTML=D.nodes.filter(x=>x.label.toLowerCase().includes(q)||x.kind.includes(q)).slice(0,350).map(x=>`<tr><td>${esc(cap(x.kind))}</td><td>${esc(x.label)}</td><td>${esc(x.id)}</td></tr>`).join('')||'<tr><td colspan="3">No matching objects</td></tr>';}document.getElementById('find').oninput=searchNodes;searchNodes();
const kinds={workbook:'#19a3a6',dashboard:'#2d8bbd',worksheet:'#4e72a9',datasource:'#9669b0',field:'#526b80','field-reference':'#526b80',connection:'#d08b55'};
function drawGraph(){const svg=document.getElementById('lineageSvg');let kind=document.getElementById('graphKind').value,q=document.getElementById('graphSearch').value.toLowerCase();let focus=D.nodes.filter(x=>(kind==='all'?['workbook','dashboard','worksheet','datasource','connection'].includes(x.kind):x.kind===kind)&&(!q||x.label.toLowerCase().includes(q))).slice(0,75);let selected=new Set(focus.map(x=>x.id));let edges=D.edges.filter(e=>selected.has(e.from)&&selected.has(e.to));let h=Math.max(450,Math.ceil(focus.length/4)*95+70);svg.setAttribute('viewBox',`0 0 1000 ${h}`);svg.style.height=h+'px';let positions={};focus.forEach((n,i)=>positions[n.id]={x:20+(i%4)*245,y:45+Math.floor(i/4)*95});let html=`<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0 0 L7 3 L0 6" fill="none" stroke="#6ca8b6"/></marker></defs>`;edges.forEach(e=>{let a=positions[e.from],b=positions[e.to];html+=`<path d="M${a.x+195} ${a.y+20} L${b.x} ${b.y+20}" stroke="#6ca8b6" opacity=".65" fill="none" marker-end="url(#arrow)"/>`;});focus.forEach(n=>{let p=positions[n.id];html+=`<g class="node" data-id="${esc(n.id)}"><rect x="${p.x}" y="${p.y}" width="195" height="45" rx="7" fill="${kinds[n.kind]||'#607080'}"/><text x="${p.x+9}" y="${p.y+17}">${esc(n.kind)}</text><text x="${p.x+9}" y="${p.y+33}">${esc(n.label.slice(0,29))}</text></g>`;});svg.innerHTML=html;svg.querySelectorAll('g.node').forEach(el=>el.onclick=()=>{let n=D.nodes.find(x=>x.id===el.dataset.id);let relations=D.edges.filter(e=>e.from===n.id||e.to===n.id);document.getElementById('nodeDetails').innerHTML=`<h3>${esc(n.label)}</h3><p class="muted">${esc(n.kind)} | ${esc(n.id)}</p><pre>${esc(JSON.stringify(n.detail,null,2))}</pre><p>${relations.length} directly recorded relationships</p>`;});}
document.getElementById('graphKind').onchange=drawGraph;document.getElementById('graphSearch').oninput=drawGraph;
let ds=D.tables.twb_datasources;let select=document.getElementById('dsSelect');select.innerHTML=ds.map((x,i)=>`<option value="${i}">${esc(x.caption||x.name)}</option>`).join('');function dsRender(){let d=ds[+select.value];if(!d){document.getElementById('dsDetails').textContent='No Tableau data sources available';return;}document.getElementById('dsDetails').innerHTML=`<h3>${esc(d.caption||d.name)}</h3><p class="muted">Tableau data source ID: ${esc(d.name)}</p>`;let rows=D.tables.twb_fields.filter(f=>f.datasource===d.name);document.getElementById('fieldTable').innerHTML=`<table><thead><tr><th>Logical field</th><th>Physical/internal name</th><th>Data type</th><th>Role</th></tr></thead><tbody>${rows.map(f=>`<tr><td>${esc(f.caption||f.name)}</td><td>${esc(f.name)}</td><td>${esc(f.datatype)}</td><td>${esc(f.role)}</td></tr>`).join('')}</tbody></table>`;}select.onchange=dsRender;dsRender();
function logicRender(){let q=document.getElementById('logicSearch').value.toLowerCase(),items=D.transformations.filter(x=>(x.name+' '+x.formula).toLowerCase().includes(q)).slice(0,200);document.getElementById('logicList').innerHTML=items.length?items.map(x=>`<details><summary>${esc(x.name)} <span class="chip">${esc(x.kind)}</span></summary><pre>${esc(x.formula)}</pre><p class="muted">Source: ${esc(x.source)}</p></details>`).join(''):'<p>No matching transformation records. BigQuery SQL / dbt / Dataform not connected.</p>';}document.getElementById('logicSearch').oninput=logicRender;logicRender();
let keys=Object.keys(D.tables);let tsel=document.getElementById('tableSelect');tsel.innerHTML=keys.map(x=>`<option>${esc(x)}</option>`).join('');function tableRender(){let rows=D.tables[tsel.value]||[],q=document.getElementById('tableSearch').value.toLowerCase();rows=rows.filter(x=>JSON.stringify(x).toLowerCase().includes(q)).slice(0,250);let columns=Array.from(new Set(rows.flatMap(Object.keys))).slice(0,14);document.getElementById('tableResults').innerHTML=`<p class="muted">Showing up to 250 matching records; source rows are preserved in the embedded metadata snapshot.</p><table><thead><tr>${columns.map(c=>`<th>${esc(c)}</th>`).join('')}</tr></thead><tbody>${rows.map(r=>`<tr>${columns.map(c=>`<td>${esc(String(r[c]||'').slice(0,350))}</td>`).join('')}</tr>`).join('')}</tbody></table>`;}tsel.onchange=tableRender;document.getElementById('tableSearch').oninput=tableRender;tableRender();
document.getElementById('limitations').innerHTML=D.limitations.map(x=>`<div class="notice">${esc(x)}</div>`).join('');document.getElementById('tableCounts').innerHTML=Object.entries(D.counts).map(([k,v])=>`<span class="chip">${esc(k)}: ${v}</span>`).join('');
</script></body></html>'''
    target=out/(re.sub(r'[^\w.-]+','_',title).strip('_')[:90]+'_Metadata_Explorer.html')
    target.write_text(html_template.replace('__DATA__',payload),encoding='utf-8')
    return str(target)



# --- Workflow 4: optional offline interactive HTML Explorer ---
import os as _wf4_os
_original_documentation_run = run
def run(input_path, output_dir='tableau_documentation_output', template_path=None, report_title=None, output_mode=None):
    """Backward compatible Word/HTML documentation plus interactive visual explorer.

    output_mode: 'word', 'html', or 'both' (default).
    """
    mode = str(output_mode or _wf4_os.environ.get('WF4_OUTPUT_MODE', 'both')).strip().lower()
    if mode not in ('word', 'html', 'both'):
        raise ValueError('output_mode must be word, html, or both')
    result = _original_documentation_run(input_path, output_dir, template_path, report_title)
    if mode in ('html', 'both'):
        explorer = build_explorer(input_path, output_dir, report_title)
        result['explorer'] = explorer
        result['visual_explorer'] = explorer
    if mode == 'html':
        from pathlib import Path as _WFPath
        _WFPath(result['word']).unlink(missing_ok=True)
        result['word'] = None
    if mode == 'word':
        from pathlib import Path as _WFPath
        _WFPath(result['html']).unlink(missing_ok=True)
        result['html'] = None
    # Reconcile the generated manifest to actual deliverables.
    manifest_path = Path(result['manifest'])
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    m['output_mode'] = mode
    m['outputs'] = {Path(v).name:{'sha256':sha256(Path(v)),'bytes':Path(v).stat().st_size} for k,v in result.items() if k in ('word','html','explorer') and v and Path(v).is_file()}
    m['visual_explorer'] = {'included':mode in ('html','both'),'offline':True,'bigquery_connected':False}
    manifest_path.write_text(json.dumps(m,indent=2),encoding='utf-8')
    return result


## Configure and generate

Set paths below, then run all cells. This notebook does not parse command-line arguments.
The input ZIP must include the packaged `.twbx` to resolve physical BigQuery connections.

In [ ]:
from pathlib import Path

# Windows example: Path(r"C:\Users\dloh\Downloads\HKSE.zip")
# Relative example: Path("HKSE.zip")
INPUT_PATH = Path(r"SC-04.zip")
OUTPUT_DIR = Path(r"tableau_documentation_output")

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input not found: {INPUT_PATH.resolve()}\n"
        "Set INPUT_PATH to the full path of your HKSE ZIP or extracted metadata folder."
    )

results = run(INPUT_PATH, OUTPUT_DIR, output_mode="both")
print("DOCX:", results["word"])
print("HTML:", results["html"])
print("Manifest:", results["manifest"])
print("Visual Explorer:", results.get("explorer"))

In [ ]:
import json
manifest=json.loads(Path(results['manifest']).read_text())
assert not manifest['full_record_appendix']
assert not manifest['word_appendices_included']
assert manifest['screenshots_embedded'] == len(manifest['screenshot_files'])
print('Validation passed:',manifest['connection_name_resolution'])
print('Friendly data-source mappings:', len(DISPLAY_DATASOURCE_NAMES))
print("Filter reference resolutions:", manifest.get("logical_field_name_resolution", {}).get("filter_reference_resolution"))
